### Import libraries
Spark functions, Window (for ranking rows within groups) and DeltaTable (for merging).

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

### Load utilities
Brings in shared schema names, S3 path and the table name helper.

In [0]:
%run ../../01_setup/utilities

### Utilities
Shared settings (catalog, schemas, S3 path) and a helper for full table names. Loaded by every pipeline notebook with `%run`.

### Notebook parameters
Widgets make the notebook reusable: catalog and data source can be changed without editing code.

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"{s3_base_path}/{data_source}/*.csv"
print(base_path)

s3://sportsbar-data-lake-vkt/gross_price/*.csv


## Bronze

### Read raw CSVs from S3
Loads the files as-is, adding load time, source file name and file size for traceability.

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

df.printSchema()
display(df.limit(10))

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-11-04T14:48:38.301Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-11-04T14:48:38.301Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-11-04T14:48:38.301Z,gross_price.csv,2741


### Save to bronze

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(table_name(bronze_schema, data_source))
)

## Silver

### Load bronze data
Also shows how many price rows each product has, to spot gaps or duplicates.

In [0]:
df_bronze = spark.table(table_name(bronze_schema, data_source))
df_bronze.printSchema()
display(df_bronze.limit(20))

display(df_bronze.groupBy("product_id").count().orderBy("product_id"))

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-11-04T14:48:47.970Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-11-04T14:48:47.970Z,gross_price.csv,2741


product_id,count
25891101,5
25891102,6
25891103,6
25891201,5
25891202,5
25891203,4
25891301,6
25891302,5
25891303,6
25891401,5


### 1. Standardise the month column
Dates arrive in four different formats; each is tried in turn and the first that works is kept.

In [0]:
df_bronze.select("month").distinct().show(truncate=False)

month_text = F.col("month").cast("string")

df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(month_text, "yyyy/MM/dd"),
        F.try_to_date(month_text, "dd/MM/yyyy"),
        F.try_to_date(month_text, "yyyy-MM-dd"),
        F.try_to_date(month_text, "dd-MM-yyyy"),
    )
)

df_silver.select("month").distinct().orderBy("month").show()

# Check: rows whose date couldn't be read (should be 0)
print("Unreadable months:", df_silver.filter(F.col("month").isNull()).count())

+----------+
|month     |
+----------+
|2025/07/01|
|01/08/2025|
|2025/09/01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025/11/01|
|2025/08/01|
|01-09-2025|
|2025/10/01|
|01/12/2025|
|01/09/2025|
|01-12-2025|
|01-08-2025|
|01/10/2025|
+----------+

+----------+
|     month|
+----------+
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
+----------+

Unreadable months: 0


### Check for duplicate prices
Same product and same month appearing more than once.

In [0]:
display(
    df_silver.groupBy("product_id", "month").count()
    .filter(F.col("count") > 1)
    .orderBy("product_id", "month")
)

product_id,month,count
88888888,2025-10-01,2
88888888,2025-11-01,2
88888888,2025-12-01,3
99999999,2025-09-01,2


### 2. Clean gross_price
Valid numbers become decimals, negative prices are made positive, and non-numeric values become 0.

In [0]:
price_text = F.col("gross_price").cast("string")

df_silver = df_silver.withColumn(
    "gross_price",
    F.when(price_text.rlike(r"^-?\d+(\.\d+)?$"), F.abs(price_text.cast("double")))
     .otherwise(F.lit(0.0))
)

# Check: how many prices were set to 0
print("Prices set to 0:", df_silver.filter(F.col("gross_price") == 0).count())
display(df_silver.limit(10))

Prices set to 0: 6


product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025-07-01,84.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,2025-08-01,0.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,2025-09-01,84.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,2025-10-01,83.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,2025-11-01,83.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
88888888,2025-12-01,83.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-07-01,68.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-08-01,68.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-09-01,68.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891102,2025-10-01,69.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741


In [0]:
display(df_silver.filter(F.col("gross_price") != F.floor(F.col("gross_price"))))

product_id,month,gross_price,read_timestamp,file_name,file_size


### 3. Attach product_code
Looks up each product_id in silver products to get the product_code used in the merged tables.

In [0]:
df_silver = df_silver.withColumn("product_id", F.col("product_id").cast("string"))

df_products = spark.table(table_name(silver_schema, "products")).select("product_id", "product_code")

# Check: price rows whose product_id has no match in products (these would be dropped)
df_unmatched = df_silver.join(df_products, "product_id", "left_anti")
print("Price rows without a matching product:", df_unmatched.count())
display(df_unmatched)

df_joined = (
    df_silver
    .join(df_products, "product_id", "inner")
    .select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")
)

display(df_joined.limit(5))

Price rows without a matching product: 18


product_id,month,gross_price,read_timestamp,file_name,file_size
88888888,2025-12-01,83.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
99999999,2025-10-01,108.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
88888888,2025-12-01,100.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
99999999,2025-09-01,132.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
88888888,2025-11-01,138.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
88888888,2025-10-01,300.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
99999999,2025-08-01,74.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
77777777,2025-07-01,49.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891502,2025-07-01,203.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891502,2025-08-01,203.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741


product_id,product_code,month,gross_price,read_timestamp,file_name,file_size
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-07-01,84.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-08-01,0.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-09-01,84.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-10-01,83.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-11-01,83.0,2026-11-04T14:48:47.970Z,gross_price.csv,2741


In [0]:
display(df_unmatched.groupBy("product_id").count().orderBy("product_id"))

product_id,count
25891502,5
77777777,1
88888888,8
99999999,4


In [0]:
# Is 25891502 in the raw products file at all?
display(spark.table(table_name(bronze_schema, "products"))
        .filter(F.col("product_id").cast("string").contains("25891502")))

product_name,product_id,category,read_timestamp,file_name,file_size


In [0]:
# Which product got the replacement ID 999999, and what was its original ID?
display(spark.table(table_name(bronze_schema, "products"))
        .filter(~F.col("product_id").cast("string").rlike("^[0-9]+$") | F.col("product_id").isNull()))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Oats Cookie Bites ChocoChip (350g),XYZ123,healthy snacks,2026-11-04T06:45:31.868Z,products.csv,1388


### Save to silver

In [0]:
(
    df_joined.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(table_name(silver_schema, data_source))
)

In [0]:
print("Rows in silver:", spark.table(table_name(silver_schema, data_source)).count())

Rows in silver: 90


## Gold

### Keep business columns and save to gold
Monthly prices per product, saved as `sb_dim_gross_price`.

In [0]:
df_gold = spark.table(table_name(silver_schema, data_source)).select("product_code", "month", "gross_price")

(
    df_gold.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(table_name(gold_schema, f"sb_dim_{data_source}"))
)

display(df_gold.limit(20))

product_code,month,gross_price
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-07-01,84.0
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-08-01,0.0
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-09-01,84.0
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-10-01,83.0
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-11-01,83.0
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-07-01,68.0
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-08-01,68.0
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-09-01,68.0
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-10-01,69.0
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-11-01,69.0


## Merge into parent

### Convert monthly prices to one price per year
Atlikon keeps one price per product per year. For each product and year, the latest non-zero monthly price is used.

In [0]:
df_price = (
    spark.table(table_name(gold_schema, "sb_dim_gross_price"))
    .withColumn("year", F.year("month"))
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
)

w = Window.partitionBy("product_code", "year").orderBy(F.col("is_zero"), F.col("month").desc())

df_latest_price = (
    df_price
    .withColumn("rnk", F.row_number().over(w))
    .filter(F.col("rnk") == 1)
)

display(df_latest_price)

product_code,month,gross_price,year,is_zero,rnk
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025-12-01,281.0,2025,0,1
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,2025-10-01,100.0,2025,0,1
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-12-01,86.0,2025,0,1
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,2025-12-01,108.0,2025,0,1
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025-12-01,493.0,2025,0,1
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025-12-01,187.0,2025,0,1
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025-11-01,440.0,2025,0,1
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025-12-01,296.0,2025,0,1
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025-12-01,50.0,2025,0,1
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,2025-12-01,138.0,2025,0,1


In [0]:
display(
    df_price.filter(F.col("gross_price") > 0)
    .groupBy("product_code")
    .agg(F.min("gross_price").alias("min_price"), F.max("gross_price").alias("max_price"))
    .filter(F.col("min_price") != F.col("max_price"))
)

product_code,min_price,max_price
e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,83.0,84.0
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,68.0,69.0
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,85.0,86.0
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,94.0,100.0
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,132.0,138.0
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,456.0,493.0
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,275.0,300.0
c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,369.0,432.0
da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05,74.0,79.0
fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,47.0,53.0


### Match Atlikon's columns and types
Renames gross_price to price_inr and uses the same column types as Atlikon's `dim_gross_price`.

In [0]:
target_name = table_name(gold_schema, "dim_gross_price")
target_types = dict(spark.table(target_name).dtypes)
print("Atlikon column types:", target_types)

df_child_price = df_latest_price.select(
    "product_code",
    F.col("gross_price").cast(target_types["price_inr"]).alias("price_inr"),
    F.col("year").cast(target_types["year"]).alias("year"),
)

df_child_price.printSchema()
display(df_child_price.limit(5))

Atlikon column types: {'product_code': 'string', 'price_inr': 'bigint', 'year': 'bigint'}
root
 |-- product_code: string (nullable = true)
 |-- price_inr: long (nullable = true)
 |-- year: long (nullable = true)



product_code,price_inr,year
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,281,2025
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,100,2025
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,86,2025
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,108,2025
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,493,2025


### Check for code clashes
A matching code would overwrite an Atlikon price. Should be 0.

In [0]:
parent_codes = spark.table(target_name).select("product_code").distinct()
clashes = parent_codes.intersect(df_child_price.select("product_code"))
print("Clashing codes:", clashes.count())

Clashing codes: 0


### Merge into Atlikon's dim_gross_price
Matches on product AND year, because the table holds one row per product per year.

In [0]:
target = DeltaTable.forName(spark, target_name)

(
    target.alias("target")
    .merge(
        source=df_child_price.alias("source"),
        condition="target.product_code = source.product_code AND target.year = source.year",
    )
    .whenMatchedUpdate(set={"price_inr": "source.price_inr"})
    .whenNotMatchedInsert(values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year",
    })
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Verify the merge
Atlikon's 794 price rows plus one row per SportsBar product per year.

In [0]:
dim_gross_price = spark.table(target_name)
print("Total price rows:", dim_gross_price.count())
display(dim_gross_price.groupBy("year").count().orderBy("year"))

Total price rows: 811


year,count
2024,397
2025,414


In [0]:
print("SportsBar price rows:", df_child_price.count())

SportsBar price rows: 17
